# Momentum, honestly backtested

**pyportfolios.com research** · 11 SPDR sector ETFs, Jan 2005 – Dec 2024 · NumPy · Pandas · Matplotlib · yfinance

A 12-1 cross-sectional momentum book on the S&P sector universe with every
look-ahead gap closed:

1. rank each month-end on the trailing **t-12..t-2** compounded return (skip the last month),
2. long the top 3 sectors / short the bottom 3, equal-weighted, dollar-neutral,
3. lag the weights exactly once, and
4. charge 10 bp (and 20 bp) on every unit of traded notional.

Real data, real costs — the point of the exercise is what survives them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.rcParams["figure.figsize"] = (10, 5)
TOP_N, COST_BP = 3, 10

## 1 · Data: the 11 SPDR sector ETFs

XLRE and XLC launch mid-sample (2015 / 2018). We keep them: a sector enters the
rankable universe only once it has a full 12-month history — the point-in-time
discipline the article insists on.

In [ ]:
tickers = ["XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY"]
px = yf.download(tickers, start="2005-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"]
mret = px.resample("ME").last().pct_change()
mret.tail(3).round(4)

## 2 · The 12-1 signal with the skip made explicit

`rolling(11)` compounds months `t-10..t`; the `.shift(1)` moves the window to
`t-11..t-1` so the most recent month is skipped (short-term reversal pollutes it).
Relative to the month we actually hold, that window is lags 2..12 — the classic 12-1.

In [ ]:
mom = ((1 + mret).rolling(11).apply(np.prod, raw=True) - 1).shift(1)

nsec  = mom.notna().sum(axis=1)          # point-in-time universe size
ranks = mom.rank(axis=1)                 # 1 = worst .. nsec = best

## 3 · The book: long top 3, short bottom 3, one explicit lag

Weights formed at month-end `t` earn returns over `t+1`. The single `.shift(1)`
below is the whole anti-look-ahead discipline.

In [ ]:
longs  = ranks.sub(nsec, axis=0).ge(-(TOP_N - 1)).astype(float)
shorts = ranks.le(TOP_N).astype(float)
valid  = nsec >= 2 * TOP_N
longs, shorts = longs[valid].fillna(0), shorts[valid].fillna(0)

w = longs.div(longs.sum(axis=1), axis=0) - shorts.div(shorts.sum(axis=1), axis=0)
w = w.reindex(mret.index).dropna(how="all")

port = (w.shift(1) * mret).sum(axis=1)[w.index[1]:]     # weights at t earn t+1
turnover = w.diff().abs().sum(axis=1) / 2               # one-way, per side
net = port - (turnover * COST_BP / 1e4 * 2).shift(1)[port.index]

def sharpe(x): return np.sqrt(12) * x.mean() / x.std(ddof=1)
print(f"gross Sharpe {sharpe(port.dropna()):.2f}   net Sharpe {sharpe(net.dropna()):.2f}")
print(f"avg monthly one-way turnover {turnover.mean():.0%}")

## 4 · The rank spread — the momentum signature

Average next-month return by momentum rank. On real sectors it is noisier than
the textbook decile chart (11 names, not 3,000 stocks) but the winners-minus-losers
tilt is visible — and it was earned without touching future data.

In [ ]:
pctrank = ranks.sub(1).div(nsec - 1, axis=0)
binidx  = (pctrank * 10).round()
stacked = pd.DataFrame({"bin": binidx.stack(),
                        "fwd": mret.shift(-1).stack().reindex(binidx.stack().index)}).dropna()
rank_avg = stacked.groupby("bin")["fwd"].mean() * 100

plt.bar(rank_avg.index + 1, rank_avg.values,
        color=["#4a4a42"] * 8 + ["#0a8a8a"] * 3)
plt.title("Average next-month return by momentum rank (1 = losers)")
plt.xlabel("rank bin"); plt.ylabel("% / month");

## 5 · Equity, costs, and the crash

Momentum's dark side is visible in the equity curve: the book compounds slowly
and then gets hit in sharp rebounds (2009, 2020), when the shorted losers rip
higher. Costs shave the rest.

In [ ]:
net20 = port - (turnover * 20 / 1e4 * 2).shift(1)[port.index]

for label, ret in [("gross", port), ("net 10bp", net), ("net 20bp", net20)]:
    eq = (1 + ret.fillna(0)).cumprod()
    yrs = len(ret.dropna()) / 12
    cagr = eq.iloc[-1] ** (1 / yrs) - 1
    mdd = (eq / eq.cummax() - 1).min()
    print(f"{label:>9}: CAGR {cagr:6.2%}  vol {ret.std()*np.sqrt(12):6.2%}  "
          f"Sharpe {sharpe(ret.dropna()):5.2f}  maxDD {mdd:6.1%}")

(1 + port.fillna(0)).cumprod().plot(label="gross")
(1 + net.fillna(0)).cumprod().plot(label="net 10bp")
plt.legend(); plt.title("12-1 sector momentum, long-short equity");

## Takeaways

- The 12-1 sector book is real but thin: the spread survives 10 bp costs
  with a visibly lower Sharpe, and 20 bp takes another bite.
- The worst month is a *momentum crash* — a violent rebound month where the
  short book of beaten-down sectors rips higher.
- Every number here is out-of-sample in the only sense that matters: signals
  use information through `t`, returns accrue at `t+1`, the universe is
  point-in-time.

*© pyportfolios.com — runnable companion to the article. Data: Yahoo Finance via yfinance.*